# Nova Geo Marts EDA

This notebook reads the geography-focused dbt marts from BigQuery into pandas DataFrames and performs lightweight EDA before any demand modeling work.

Primary tables:

- `mart_geo_market_opportunity`: market-level opportunity ranking
- `mart_geo_market_category_day_features`: market-category-day modeling feature table
- `mart_geo_weather_category_sensitivity`: market-category weather sensitivity summary

Power BI should ultimately read curated BigQuery/dbt tables. This notebook is for exploration and model preparation only.

## 1. Setup

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
import plotly.express as px
from google.cloud import bigquery

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.4f}".format)

px.defaults.template = "plotly_white"
px.defaults.width = 1_050
px.defaults.height = 560

## 2. BigQuery Connector

This uses your active Google Application Default Credentials or the credentials available in the notebook environment. If authentication fails, run `gcloud auth application-default login` outside the notebook and retry.

In [2]:
PROJECT_ID = "nova-project-498911"
DATASET_ID = "dbt_doruk"

TABLES = {
    "market_opportunity": "mart_geo_market_opportunity",
    "market_category_day_features": "mart_geo_market_category_day_features",
    "weather_category_sensitivity": "mart_geo_weather_category_sensitivity",
}

client = bigquery.Client(project=PROJECT_ID)


def table_ref(table_name: str) -> str:
    return f"`{PROJECT_ID}.{DATASET_ID}.{table_name}`"


def read_bq(sql: str) -> pd.DataFrame:
    return client.query(sql).to_dataframe()


dataset = client.get_dataset(f"{PROJECT_ID}.{DATASET_ID}")
print(f"Connected to {dataset.full_dataset_id}")

Connected to nova-project-498911:dbt_doruk


## 3. Read Geo Marts

In [3]:
market_opportunity = read_bq(
    f"""
    select *
    from {table_ref(TABLES["market_opportunity"])}
    order by opportunity_rank
    """
)

market_category_day_features = read_bq(
    f"""
    select *
    from {table_ref(TABLES["market_category_day_features"])}
    """
)

weather_category_sensitivity = read_bq(
    f"""
    select *
    from {table_ref(TABLES["weather_category_sensitivity"])}
    order by category, category_weather_sensitivity_rank
    """
)

dfs = {
    "market_opportunity": market_opportunity,
    "market_category_day_features": market_category_day_features,
    "weather_category_sensitivity": weather_category_sensitivity,
}

for name, df in dfs.items():
    print(f"{name:<32} {df.shape[0]:>8,} rows x {df.shape[1]:>3} columns")

/Users/doruk/dev/nova-analytics/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


market_opportunity                     16 rows x  76 columns
market_category_day_features       29,280 rows x  80 columns
weather_category_sensitivity           80 rows x  28 columns


In [4]:
for df in [market_opportunity, market_category_day_features, weather_category_sensitivity]:
    for col in df.columns:
        if col.endswith("_date") or col == "order_date":
            df[col] = pd.to_datetime(df[col])

market_category_day_features["year_month"] = market_category_day_features["order_date"].dt.to_period("M").astype(str)

## 4. Table Health Checks

In [5]:
health_rows = []
for name, df in dfs.items():
    health_rows.append(
        {
            "table": name,
            "rows": len(df),
            "columns": df.shape[1],
            "duplicate_rows": int(df.duplicated().sum()),
            "total_null_cells": int(df.isna().sum().sum()),
        }
    )

table_health = pd.DataFrame(health_rows)
table_health

,table,rows,columns,duplicate_rows,total_null_cells
0,market_opportunity,16,76,0,0
1,market_category_day_features,29280,80,0,0
2,weather_category_sensitivity,80,28,0,0


In [6]:
grain_checks = pd.DataFrame(
    [
        {
            "table": "market_opportunity",
            "expected_grain": "market_id",
            "rows": len(market_opportunity),
            "distinct_grain": market_opportunity[["market_id"]].drop_duplicates().shape[0],
        },
        {
            "table": "market_category_day_features",
            "expected_grain": "market_id + category + order_date",
            "rows": len(market_category_day_features),
            "distinct_grain": market_category_day_features[["market_id", "category", "order_date"]].drop_duplicates().shape[0],
        },
        {
            "table": "weather_category_sensitivity",
            "expected_grain": "market_id + category",
            "rows": len(weather_category_sensitivity),
            "distinct_grain": weather_category_sensitivity[["market_id", "category"]].drop_duplicates().shape[0],
        },
    ]
)

grain_checks["grain_ok"] = grain_checks["rows"] == grain_checks["distinct_grain"]
grain_checks

,table,expected_grain,rows,distinct_grain,grain_ok
0,market_opportunity,market_id,16,16,True
1,market_category_day_features,market_id + category + order_date,29280,29280,True
2,weather_category_sensitivity,market_id + category,80,80,True


In [7]:
market_category_day_features[["order_date", "market_id", "category"]].agg(
    {
        "order_date": ["min", "max", "nunique"],
        "market_id": "nunique",
        "category": "nunique",
    }
)

,order_date,market_id,category
min,2024-01-01 00:00:00,NaN,NaN
max,2024-12-31 00:00:00,NaN,NaN
nunique,366,16.0000,5.0000


## 5. Market Opportunity Overview

In [8]:
market_cols = [
    "opportunity_rank",
    "market_name",
    "nova_region",
    "opportunity_segment",
    "opportunity_score",
    "current_performance_score",
    "macro_potential_score",
    "supply_score",
    "adoption_score",
    "total_transactions",
    "total_gmv_usd",
    "annual_active_users",
    "country_annual_active_users",
    "country_active_user_penetration",
    "country_urban_active_user_penetration",
    "transaction_growth_rate",
    "completion_rate",
]

market_opportunity[market_cols]

,opportunity_rank,market_name,nova_region,opportunity_segment,opportunity_score,current_performance_score,macro_potential_score,supply_score,adoption_score,total_transactions,total_gmv_usd,annual_active_users,country_annual_active_users,country_active_user_penetration,country_urban_active_user_penetration,transaction_growth_rate,completion_rate
0,1,Singapore,Southeast Asia,Growth candidate,71.3800,68.2491,71.5565,48.0147,100.0000,3338833,"217,194,167.9400",119016,119016,0.0197,0.0197,0.7005,0.9164
1,2,Jakarta,Southeast Asia,Maintain,53.5500,90.8554,23.0222,77.7036,2.2819,4732472,"202,985,200.9800",153953,153953,0.0005,0.0009,0.6925,0.9064
2,3,New York,North America,Maintain,51.2700,67.6889,59.5140,63.1167,0.3795,2975236,"246,429,575.5700",110470,110470,0.0003,0.0004,0.7998,0.9167
3,4,Dubai,EMEA,Maintain,48.0700,43.5307,66.4949,39.0724,41.9850,2481437,"175,831,982.6700",85675,85675,0.0078,0.0091,0.6970,0.9117
4,5,Manila,Southeast Asia,Maintain,45.6300,69.7141,15.1537,80.6785,6.5181,3873921,"177,769,906.4700",128517,128517,0.0011,0.0020,0.6977,0.9075
5,6,Tokyo,East Asia,Monitor,44.6400,55.6715,53.7388,55.6685,2.9304,2792160,"205,357,733.1000",102357,102357,0.0008,0.0009,0.7660,0.9149
6,7,London,Europe,Monitor,44.4600,65.1134,51.4260,37.1870,6.8967,2838961,"246,716,950.4400",102539,102539,0.0015,0.0018,0.7881,0.9145
7,8,Mumbai,South Asia,Monitor,43.7100,69.4541,20.0000,71.7755,0.2423,4309477,"138,944,868.0300",136641,256118,0.0002,0.0005,0.6987,0.9016
8,9,Bangkok,Southeast Asia,Monitor,41.3300,53.1907,38.0804,57.0388,8.9354,3267079,"152,273,022.5000",110905,110905,0.0015,0.0025,0.6905,0.9101
9,10,Istanbul,EMEA,Monitor,41.0300,51.1664,38.1596,62.3381,5.5563,3164862,"151,223,637.8200",110598,110598,0.0013,0.0014,0.6901,0.9085


In [9]:
fig = px.bar(
    market_opportunity.sort_values("opportunity_score"),
    x="opportunity_score",
    y="market_name",
    color="opportunity_segment",
    orientation="h",
    title="Market Opportunity Score by Market",
    labels={"market_name": "Market", "opportunity_score": "Opportunity score"},
)
fig.show()

In [10]:
fig = px.scatter_geo(
    market_opportunity,
    lat="latitude",
    lon="longitude",
    size="total_gmv_usd",
    color="opportunity_segment",
    hover_name="market_name",
    hover_data={
        "opportunity_score": ":.2f",
        "total_transactions": ":,",
        "total_gmv_usd": ":,.0f",
        "country_active_user_penetration": ":.3%",
        "transaction_growth_rate": ":.2%",
        "latitude": False,
        "longitude": False,
    },
    title="Market Opportunity Atlas",
)
fig.update_geos(projection_type="natural earth")
fig.show()

In [11]:
country_adoption = (
    market_opportunity[[
        "country_name",
        "country_iso3",
        "population_total",
        "country_annual_active_users",
        "country_active_user_penetration",
        "country_urban_active_user_penetration",
    ]]
    .drop_duplicates(subset=["country_iso3"])
    .sort_values("country_active_user_penetration", ascending=True)
)

fig = px.bar(
    country_adoption,
    x="country_active_user_penetration",
    y="country_name",
    orientation="h",
    color="country_urban_active_user_penetration",
    title="Country-Level Active User Penetration",
    labels={
        "country_active_user_penetration": "Active users / country population",
        "country_urban_active_user_penetration": "Active users / urban population",
        "country_name": "Country",
    },
    hover_data={
        "country_annual_active_users": ":,",
        "population_total": ":,",
        "country_active_user_penetration": ":.3%",
        "country_urban_active_user_penetration": ":.3%",
    },
)
fig.update_layout(xaxis_tickformat=".2%")
fig.show()

In [12]:
score_components = market_opportunity[
    [
        "market_name",
        "current_performance_score",
        "macro_potential_score",
        "supply_score",
        "adoption_score",
    ]
].melt(id_vars="market_name", var_name="score_component", value_name="score")

fig = px.bar(
    score_components,
    x="market_name",
    y="score",
    color="score_component",
    barmode="group",
    title="Opportunity Score Components",
    labels={"market_name": "Market", "score": "Score"},
)
fig.update_layout(xaxis_tickangle=-35)
fig.show()

## 6. Market-Category-Day Demand Shape

In [13]:
monthly_market = (
    market_category_day_features.groupby(["year_month", "market_name"], as_index=False)
    .agg(transactions=("transactions", "sum"), gmv_usd=("gmv_usd", "sum"), active_users=("active_users", "sum"))
)

fig = px.line(
    monthly_market,
    x="year_month",
    y="transactions",
    color="market_name",
    title="Monthly Transactions by Market",
    labels={"year_month": "Month", "transactions": "Transactions", "market_name": "Market"},
)
fig.show()

In [14]:
category_summary = (
    market_category_day_features.groupby("category", as_index=False)
    .agg(
        transactions=("transactions", "sum"),
        gmv_usd=("gmv_usd", "sum"),
        avg_daily_transactions=("transactions", "mean"),
        avg_transaction_amount_usd=("avg_transaction_amount_usd", "mean"),
        completion_rate=("completion_rate", "mean"),
        promo_share=("promo_share", "mean"),
    )
    .sort_values("transactions", ascending=False)
)

category_summary

,category,transactions,gmv_usd,avg_daily_transactions,avg_transaction_amount_usd,completion_rate,promo_share
2,Food Delivery,14864102,"429,173,083.3400","2,538.2688",30.3679,0.9384,0.2514
4,Ride Hailing,11202731,"312,249,208.7400","1,913.0347",29.3352,0.8818,0.2514
1,E-Commerce,11005827,"1,113,469,116.3400","1,879.4103",100.7122,0.8657,0.1667
3,Grocery,6915310,"520,815,770.0800","1,180.8931",76.8890,0.9155,0.0000
0,Digital Wallet,6012030,"279,577,129.5300","1,026.6445",49.3741,0.9671,0.0000


In [15]:
fig = px.bar(
    category_summary,
    x="category",
    y="transactions",
    color="category",
    title="Annual Transactions by Category",
    labels={"category": "Category", "transactions": "Transactions"},
)
fig.show()

In [16]:
market_category_mix = (
    market_category_day_features.groupby(["market_name", "category"], as_index=False)
    .agg(transactions=("transactions", "sum"))
)
market_category_mix["market_category_share"] = market_category_mix["transactions"] / market_category_mix.groupby("market_name")["transactions"].transform("sum")

fig = px.bar(
    market_category_mix,
    x="market_name",
    y="market_category_share",
    color="category",
    title="Category Mix by Market",
    labels={"market_name": "Market", "market_category_share": "Share of market transactions"},
)
fig.update_layout(xaxis_tickangle=-35, yaxis_tickformat=".0%")
fig.show()

## 7. Weather and Local Demand Checks

In [17]:
weather_market = (
    market_category_day_features.groupby("market_name", as_index=False)
    .agg(
        rain_day_share=("is_rain_day", "mean"),
        avg_temperature_c=("temperature_2m_mean_c", "mean"),
        avg_precipitation_mm=("precipitation_sum_mm", "mean"),
        avg_daily_transactions=("transactions", "mean"),
    )
    .sort_values("rain_day_share", ascending=False)
)

weather_market

,market_name,rain_day_share,avg_temperature_c,avg_precipitation_mm,avg_daily_transactions
13,Singapore,0.9754,26.7866,10.5298,"1,824.4989"
5,Jakarta,0.8579,27.3967,6.8311,"2,586.0503"
7,Manila,0.7896,28.0230,6.3082,"2,116.8967"
3,Ho Chi Minh City,0.7158,28.1260,5.8052,"1,688.8951"
1,Bangkok,0.6858,28.7918,4.4727,"1,785.2891"
6,London,0.6557,11.7238,2.4415,"1,551.3448"
14,Sydney,0.6175,17.8352,2.9183,619.4415
12,Sao Paulo,0.5656,20.3637,3.4691,"1,630.1317"
0,Bangalore,0.5601,24.0268,2.9533,"2,054.3082"
15,Tokyo,0.5383,16.6926,4.7030,"1,525.7705"


In [18]:
fig = px.scatter(
    weather_market,
    x="avg_temperature_c",
    y="rain_day_share",
    size="avg_daily_transactions",
    color="avg_precipitation_mm",
    hover_name="market_name",
    title="Weather Profile by Market",
    labels={
        "avg_temperature_c": "Average temperature, C",
        "rain_day_share": "Rain-day share",
        "avg_precipitation_mm": "Avg precipitation, mm",
    },
)
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

In [19]:
rain_lift = weather_category_sensitivity.copy()
rain_lift["rain_transaction_lift_pct"] = pd.to_numeric(rain_lift["rain_transaction_lift_pct"], errors="coerce")

fig = px.bar(
    rain_lift.sort_values("rain_transaction_lift_pct"),
    x="rain_transaction_lift_pct",
    y="market_name",
    color="rain_sensitivity_segment",
    facet_col="category",
    facet_col_wrap=2,
    title="Rain Transaction Lift by Market and Category",
    labels={"rain_transaction_lift_pct": "Rain transaction lift", "market_name": "Market"},
)
fig.update_xaxes(tickformat=".0%")
fig.show()

## 8. Notes and Modeling Candidates

Use this section to capture observations before moving to the dedicated demand-modeling notebook.

- Candidate target: `transactions`
- Secondary target: `gmv_usd`
- Candidate split: train through October 2024, validate on November-December 2024
- Candidate features: market context, calendar fields, promo share, service supply, platform mix, weather, macro indicators
- Candidate outputs to write back to BigQuery later: predictions, feature importance, model metrics, and market/category segment labels